### 1. Train FHMM Step by Step

In [ ]:
import os

# --- Kaggle perf: gioi han so luong BLAS/OpenMP cho moi process ---
# Phai dat TRUOC khi numpy/pandas duoc import lan dau, de cac worker
# do joblib (Parallel) sinh ra sau nay khong bi "oversubscribe" CPU
# (nhieu process x nhieu thread BLAS cung tranh nhau vai core cua Kaggle).
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import pandas as pd
import numpy as np

# Kaggle-friendly path setup
if os.path.exists('/kaggle/input'):
    data_dir = '/kaggle/input/nilm-refit-house2-processed'  # Update dataset name if different
    work_dir = '/kaggle/working'  # /kaggle/input chi DOC -> moi file ghi ra phai de o day
else:
    data_dir = '../../data/processed_data'  # Local fallback
    work_dir = data_dir

os.makedirs(work_dir, exist_ok=True)
print(f"data_dir (read):  {data_dir}")
print(f"work_dir (write): {work_dir}")


### Luu y toc do khi chay tren Kaggle

- **Tat GPU Accelerator** (Settings -> Accelerator -> None). `hmmlearn` (GaussianHMM) chi chay tren CPU, khong ho tro GPU/TPU -- bat GPU khong giup gi ma con khien Kaggle **cap it CPU hon** (2 core thay vi 4) va ton quota GPU hang tuan.
- Notebook da dung `joblib.Parallel` de train nhieu HMM song song tren cac core CPU ma Kaggle cap (xem cell train FHMM va cell grid search).
- Da gioi han so luong BLAS/OpenMP threads = 1 cho moi process con de tranh tranh chap CPU khi chay song song (xem cell ngay tren).


### 2. Import Libraries

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

### 3. Set Project Path

### 4. Load Data


In [ ]:
csv_path = '/kaggle/input/datasets/meowll/nilm-data/House2_full.csv'
print(f'Loading data from: {csv_path}')
df = pd.read_csv(csv_path)
print(f'Number of rows: {len(df):,}')
print(f'Number of columns: {len(df.columns)}')
print(f'Column names: {df.columns.tolist()}')


### 6. Process Time Column

In [ ]:
df['Time'] = pd.to_datetime(df['Time'])

print(f"Time data type: {df['Time'].dtype}") 
print(f"Start time: {df['Time'].min()}")
print(f"End time: {df['Time'].max()}")
print(f"Time interval: {df['Time'].max() - df['Time'].min()}")

time_diff = df['Time'].diff().dropna()
print(f"Mean distance between samples: {time_diff.mean()}")
print(f"Most common distance: {time_diff.mode()[0]}")

### 7. Calculate Appliance_Others

In [ ]:
appliance_cols = ['Appliance1', 'Appliance2', 'Appliance3', 
                  'Appliance4', 'Appliance5', 'Appliance6',
                  'Appliance7', 'Appliance8', 'Appliance9']

df['sum_9_appliances'] = df[appliance_cols].sum(axis=1)

df['Appliance_Others'] = df['Aggregate'] - df['sum_9_appliances']

negative_count = (df['Appliance_Others'] < 0).sum()
print(f"Number of rows with Appliance_Others < 0 (to be deleted): {negative_count:,}")

df = df[df['Appliance_Others'] >= 0].copy()
print(f"The {negative_count:,} line has been removed. The current minimum value is: {df['Appliance_Others'].min()}")

df.drop(columns=['sum_9_appliances'], inplace=True)

print("Appliance_Others Distribution BEFORE Filtering")
print(df['Appliance_Others'].describe(percentiles=[.01, .25, .5, .75, .99, .999]))
print(f"\nNumber of negative rows : {(df['Appliance_Others'] < 0).sum():,}")
print(f"Absolute Max: {df['Appliance_Others'].max():.1f} W")

upper_bound = df['Aggregate'].quantile(0.999)
print(f"99.9th percentile of Aggregate = {upper_bound:.1f} W")
print(f"Max Aggregate in dataset= {df['Aggregate'].max():.1f} W\n")

df['Appliance_Others'] = df['Appliance_Others'].clip(lower=0, upper=upper_bound)

print(f"Summary of Appliance_Others AFTER edge lamination")
print(df['Appliance_Others'].describe())

### 8. Add Time Features

Why are time features needed?

- Washing machines often run between 8-10 AM or 7-9 PM.

- Microwaves are often used during mealtimes (12 PM, 6 PM).

- When the model knows "it's 3 AM," it will predict fewer appliances will be turned on.

In [ ]:
df['hour'] = df['Time'].dt.hour
df['minute'] = df['Time'].dt.minute
df['dayofweek'] = df['Time'].dt.dayofweek
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

print("Time features added")
print(df[['Time', 'hour', 'minute', 'dayofweek', 'is_weekend']].head(3))

### 9. Chronological Split (70/10/20)

In [ ]:
df = df.sort_values('Time').reset_index(drop=True)

total_rows = len(df)
train_end  = int(total_rows * 0.70)
val_end    = int(total_rows * 0.80)

df_train = df.iloc[:train_end].copy()
df_val   = df.iloc[train_end:val_end].copy()
df_test  = df.iloc[val_end:].copy()  

print(f"Total rows: {total_rows:,}")
print(f"Train: {len(df_train):,} rows ({len(df_train)/total_rows*100:.1f}%)")
print(f"Val: {len(df_val):,} rows ({len(df_val)/total_rows*100:.1f}%)")
print(f"Test: {len(df_test):,} rows ({len(df_test)/total_rows*100:.1f}%)")
print(f"\nTrain: {df_train['Time'].min()} → {df_train['Time'].max()}")
print(f"Val: {df_val['Time'].min()} → {df_val['Time'].max()}")
print(f"Test: {df_test['Time'].min()} → {df_test['Time'].max()}")

assert df_train['Time'].max() < df_val['Time'].min()
assert df_val['Time'].max()   < df_test['Time'].min()
print("\nNo overlap between sets")

### 11. Separate Features and Targets

After this data splitting step:

|Variable|Function|
|-|-|
|`y_train`|`fit()` - train model|
|`agg_val`, `y_val`|grid search — select `n_states`/`n_iter`|
|`agg_test`, `y_test`|Final evaluation — OPENED ONLY ONCE|

In [ ]:
target_cols = ['Appliance1', 'Appliance2', 'Appliance3',
               'Appliance4', 'Appliance5', 'Appliance6',
               'Appliance7', 'Appliance8', 'Appliance9',
               'Appliance_Others']

y_train = df_train[target_cols]

agg_val = df_val['Aggregate'].values
y_val   = df_val[target_cols]

agg_test = df_test['Aggregate'].values
y_test  = df_test[target_cols]

print("Sizes of the sets: ")
print(f"  y_train:  {y_train.shape}")
print(f"  agg_val:  {agg_val.shape}     y_val:   {y_val.shape}")
print(f"  agg_test: {agg_test.shape}   y_test:  {y_test.shape}")


### 12. Define the FHMM Class

Factorial HMM uses mean-field approximation.

Train K independent HMMs (K = number of devices).

Idea:

- Each device = 1 separate HMM

- Train each HMM at its own power level

- When predicting: use Aggregate to find the best hidden state

Mean-Field Residual Inference (Coordinate Descent) - Predict the power output of each appliance from Aggregate

- Loop 1, appliance_A (refrigerator):

    - other = appliance_B_est + appliance_C_est + ... = 2300W

    - resid  = agg - other = 2500 - 2300 = 200W

    - states = HMM_refrigerator.predict([200W])

    - refrigerator_est = 150W  (state ON)

- Repeat this process for each appliance, then repeat it twice to complete the cycle.

In [ ]:
import importlib.util
if importlib.util.find_spec('hmmlearn') is None:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'hmmlearn'])

from hmmlearn import hmm
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

OTHERS_COL = 'Appliance_Others'

class SimpleFHMM:
    def __init__(self, n_states_per_appliance: dict = None, n_states: int = 2, n_iter: int = 100):
        self.n_states_per_appliance = n_states_per_appliance or {}
        self.n_states = n_states
        self.n_iter = n_iter
        self.models = {}
        self.is_fitted = False
    
    def _get_n_states(self, col_name):
        return self.n_states_per_appliance.get(col_name, self.n_states)
    
    def fit(self, y_train: pd.DataFrame):
        # Chỉ train HMM cho các thiết bị THỰC (bỏ qua Appliance_Others)
        train_cols = [c for c in y_train.columns if c != OTHERS_COL]
        print(f"Training {len(train_cols)} HMM models (excluding {OTHERS_COL})...")
        
        def train_single_hmm(col_name, obs_data, n_states, n_iter):
            model = hmm.GaussianHMM(
                n_components=n_states,
                covariance_type="diag",
                n_iter=n_iter,
                random_state=42
            )
            model.fit(obs_data)
            return col_name, model

        results = Parallel(n_jobs=-1, verbose=5)(  # -1 = dung TOAN BO core CPU Kaggle cap (batch job, khong can de chua 1 core)
            delayed(train_single_hmm)(
                col, y_train[col].values.reshape(-1, 1),
                self._get_n_states(col), self.n_iter
            )
            for col in train_cols
        )

        for col, trained_model in results:
            self.models[col] = trained_model
            means = sorted(trained_model.means_.flatten().round(1))
            print(f"Done [{col}] (n_states={trained_model.n_components}): {means}")
        
        self.is_fitted = True
        print("\nTraining complete!")
        return self
  
    def predict(self, agg_power: np.ndarray, n_coord_iter: int = 5) -> pd.DataFrame:
        if not self.is_fitted:
            raise RuntimeError("Not yet trained! Call fit() first.")

        cols = list(self.models.keys())  # Chỉ có 9 thiết bị thực
        N = len(agg_power)

        pred_power = {
            col: np.full(N, max(self.models[col].means_.min(), 0.0))
            for col in cols
        }

        for _iter in range(n_coord_iter):
            for col in cols:
                model = self.models[col]
                other_power = sum(pred_power[c] for c in cols if c != col)
                residual = (agg_power - other_power).clip(min=0)
                states = model.predict(residual.reshape(-1, 1))
                pred_power[col] = np.array(
                    [model.means_[s, 0] for s in states]
                ).clip(min=0)

        # Appliance_Others = Aggregate - tổng 9 thiết bị (luôn >= 0)
        sum_9 = sum(pred_power[c] for c in cols)
        pred_power[OTHERS_COL] = (agg_power - sum_9).clip(min=0)

        return pd.DataFrame(pred_power)

### 13. Train Model

The dataset is very large, so training will take a long time. For quick testing, use the first 200,000 points.

Once the model is stable, retrain it with the full dataset.

In [ ]:
QUICK_TRAIN = True

if QUICK_TRAIN:
    y_train_subset = y_train.iloc[:200_000]
    print(f"Use {len(y_train_subset):,} instances for training (quick mode)")
else:
    y_train_subset = y_train
    print(f"Use full {len(y_train_subset):,} instances for training")

fhmm = SimpleFHMM(n_states=2, n_iter=100)
fhmm.fit(y_train_subset)

### 14. Inspect HMM Models

In [ ]:
print("Information about the trained models.")

for col, model in fhmm.models.items():
    means = sorted(model.means_.flatten()) 
    print(f"\n{col}:")
    print(f"  State means (W): {[f'{m:.1f}' for m in means]}")
    print(f"  Transition matrix:")
    for row in model.transmat_:
        print(f"    {[f'{p:.3f}' for p in row]}")

### 15. Grid Search for Hyperparameters

In [ ]:
from sklearn.metrics import mean_absolute_error
from joblib import Parallel, delayed
from itertools import product
from collections import defaultdict

CANDIDATE_STATES = [2, 3, 4, 5, 6]
N_ITER_FIXED = 100
TRAIN_SUBSET = 200_000
y_train_sub = y_train.iloc[:TRAIN_SUBSET]

tunable_cols = [c for c in target_cols if c != OTHERS_COL]

print("=" * 70)
print("GRID SEARCH - TIM n_states TOI UU CHO TUNG THIET BI (song song)")
print("=" * 70)

def _fit_and_score(col, ns):
    model = hmm.GaussianHMM(
        n_components=ns, covariance_type="diag",
        n_iter=N_ITER_FIXED, random_state=42
    )
    # Train tren du lieu CUA CHINH thiet bi do
    model.fit(y_train_sub[col].values.reshape(-1, 1))

    # Danh gia tren du lieu CUA CHINH thiet bi do (tap Val)
    # -> Do luong kha nang mo hinh hoa hanh vi thiet bi
    states = model.predict(y_val[col].values.reshape(-1, 1))
    pred = np.array([model.means_[s, 0] for s in states]).clip(min=0)

    mae = mean_absolute_error(y_val[col].values, pred)
    means = sorted(model.means_.flatten().round(1))
    return col, ns, mae, means

# 9 thiet bi x 5 candidate states = 45 to hop doc lap -> chay het
# tren joblib.Parallel mot lan thay vi vong lap tuan tu nhu truoc,
# tan dung toan bo so core CPU ma Kaggle cap.
combos = list(product(tunable_cols, CANDIDATE_STATES))
results = Parallel(n_jobs=-1, verbose=5)(
    delayed(_fit_and_score)(col, ns) for col, ns in combos
)

by_col = defaultdict(list)
for col, ns, mae, means in results:
    by_col[col].append((ns, mae, means))

best_n_states_per_appliance = {}

for col in tunable_cols:
    print(f"\n--- {col} ---")
    for ns, mae, means in sorted(by_col[col], key=lambda x: x[0]):
        print(f"  n_states={ns}  MAE={mae:.2f}W  means={means}")

    best_ns, best_mae, _ = min(by_col[col], key=lambda x: x[1])
    best_n_states_per_appliance[col] = best_ns
    print(f"  >>> BEST: n_states={best_ns}, MAE={best_mae:.2f}W")

print("\n" + "=" * 70)
print("KET QUA TONG HOP:")
for col, ns in best_n_states_per_appliance.items():
    print(f"  {col}: n_states = {ns}")


### 16. Retrain Best Configuration

In [ ]:
print("Best n_states for each appliance:")
for col, ns in best_n_states_per_appliance.items():
    print(f"  {col}: n_states={ns}")

N_ITER_FIXED = 100

print(f"\nTraining best model on entire {len(y_train):,} instances...")

best_model = SimpleFHMM(
    n_states_per_appliance=best_n_states_per_appliance,
    n_iter=N_ITER_FIXED
)

best_model.fit(y_train)  
print("Complete")

print("\nState means of best model:")
for col, model in best_model.models.items():
    means = sorted(model.means_.flatten())
    print(f"  {col}: {[f'{m:.1f}W' for m in means]}")


### 17. Test Set Predictions

In [ ]:
y_pred_df = best_model.predict(agg_test, n_coord_iter=3)

print(f"y_pred_df shape: {y_pred_df.shape}")
print(f"\nFirst 5 predictions:")
print(y_pred_df.head())

### 18. Evaluation Metrics

Metrics:

- MAE: Mean Absolute Error (W) — average error

- RMSE: Root Mean Square Error (W) — penalize larger errors.

- NDE: Normalized Disaggregation Error — standardized error

- SAE: Signal Aggregate Error — total energy error
    
- F1:  On/Off detection accuracy

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def compute_nilm_metrics(y_true_df: pd.DataFrame, 
                          y_pred_df: pd.DataFrame) -> pd.DataFrame:

    results = {}
    
    for col in y_true_df.columns:
        y_true = y_true_df[col].values
        y_pred = y_pred_df[col].values

        mae = mean_absolute_error(y_true, y_pred)

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        nde = np.sum((y_true - y_pred)**2) / (np.sum(y_true**2) + 1e-8)

        sae = abs(y_pred.sum() - y_true.sum()) / (y_true.sum() + 1e-8)

        threshold = 10  # W

        y_true_on = (y_true > threshold).astype(int)
        y_pred_on = (y_pred > threshold).astype(int)
        tp = ((y_true_on == 1) & (y_pred_on == 1)).sum()
        fp = ((y_true_on == 0) & (y_pred_on == 1)).sum()
        fn = ((y_true_on == 1) & (y_pred_on == 0)).sum()
        precision = tp / (tp + fp + 1e-8)
        recall    = tp / (tp + fn + 1e-8)
        f1        = 2 * precision * recall / (precision + recall + 1e-8)
        
        results[col] = {
            'MAE (W)':  round(mae, 2),
            'RMSE (W)': round(rmse, 2),
            'NDE':      round(nde, 4),
            'SAE':      round(sae, 4),
            'F1':       round(f1, 4),
        }
    
    return pd.DataFrame(results).T

metrics_df = compute_nilm_metrics(y_test, y_pred_df)
print("\nFHMM MODEL RATING TABLE:")
print(metrics_df.to_string())

### 19. Visualization Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

# Đã bỏ .iloc[WINDOW_SIZE:] vì df_test giờ được lấy nguyên vẹn
time_test = df_test['Time'].reset_index(drop=True)

N_PLOT = min(32_400, len(time_test))

all_appliances = ['Appliance1', 'Appliance2', 'Appliance3',
                  'Appliance4', 'Appliance5', 'Appliance6',
                  'Appliance7', 'Appliance8', 'Appliance9',
                  'Appliance_Others']

colors_actual = '#1565C0'  
colors_pred   = '#E53935' 

fig, axes = plt.subplots(5, 2, figsize=(20, 28))
axes = axes.flatten()  # chuyển từ 2D array sang 1D

for i, col in enumerate(all_appliances):
    ax = axes[i]
    
    t = time_test[:N_PLOT]
    actual = y_test[col].values[:N_PLOT]
    predicted = y_pred_df[col].values[:N_PLOT]
    
    ax.plot(t, actual, 
            color=colors_actual, linewidth=0.8, alpha=0.8,
            label='Reality', zorder=2)
    
    ax.plot(t, predicted, 
            color=colors_pred, linewidth=0.8, alpha=0.8,
            label='Prediction (FHMM)', linestyle='--', zorder=3)

    mae_val = np.abs(actual - predicted).mean()
    
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    
    ax.set_title(f'Appliance: {col}  |  MAE = {mae_val:.1f} W', 
                 fontsize=11, fontweight='bold', pad=8)
    ax.set_ylabel('Power (W)', fontsize=9)
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    ax.grid(True, alpha=0.25, linestyle='--')
    
    ax.set_facecolor('#F8F9FA')

for j in range(len(all_appliances), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(f'FHMM Disaggregation — Reality vs. Prediction\n'
             f'(n_states = {best_model.n_states})',
             fontsize=16, fontweight='bold', y=1.01)

plt.tight_layout()

output_path = 'fhmm_results.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.show()